# 03 - Strategy Backtesting

Full workflow:
1. Load data
2. Apply strategy signals
3. Run backtest
4. Analyze results
5. Compare strategies

In [ ]:
import sys
sys.path.insert(0, '/app')

import pandas as pd
import matplotlib.pyplot as plt

from data_pipeline.loader import DataLoader
from strategies.ema_crossover import EMACrossover
from strategies.mean_reversion import MeanReversion
from strategies.momentum import Momentum
from backtesting.engine import BacktestEngine
from backtesting.report import BacktestReport

plt.style.use('seaborn-v0_8-darkgrid')

## 1. Load Data

In [ ]:
loader = DataLoader()
df = loader.load_ohlcv('AAPL', start_date='2020-01-01')
print(f'Loaded {len(df)} rows from {df.index.min()} to {df.index.max()}')

## 2. EMA Crossover Strategy

In [ ]:
# Initialize strategy and backtest engine
ema_strategy = EMACrossover(fast_period=12, slow_period=26)
engine = BacktestEngine()

# Run backtest
ema_result = engine.run(ema_strategy, df, ticker='AAPL')

# Print results
report = BacktestReport()
print(report.generate(ema_result))

## 3. Mean Reversion Strategy

In [ ]:
mr_strategy = MeanReversion(lookback=20, entry_std=2.0, exit_std=0.5)
mr_result = engine.run(mr_strategy, df, ticker='AAPL')
print(report.generate(mr_result))

## 4. Momentum Strategy

In [ ]:
mom_strategy = Momentum(lookback=20)
mom_result = engine.run(mom_strategy, df, ticker='AAPL')
print(report.generate(mom_result))

## 5. Compare All Strategies

In [ ]:
comparison = report.compare([ema_result, mr_result, mom_result])
comparison

In [ ]:
# Plot equity curves
fig, ax = plt.subplots(figsize=(14, 6))

for result in [ema_result, mr_result, mom_result]:
    eq = pd.Series(result.equity_curve)
    ax.plot(eq.values, label=result.strategy_name, linewidth=1)

ax.set_title('Equity Curve Comparison')
ax.set_ylabel('Portfolio Value ($)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Save Results to Database

In [ ]:
# Register strategies and save backtest results
from strategies.registry import StrategyRegistry

strategy_reg = StrategyRegistry()

for strategy, result in [(ema_strategy, ema_result), (mr_strategy, mr_result), (mom_strategy, mom_result)]:
    strategy_reg.register(strategy, performance_metrics=result.metrics)
    report.save_to_db(result)

print('All results saved!')